# 06 Score baseline faithfulness

Score or resume 618 baseline summaries with SummaCZS and AlignScore.

**Kernel:** `.venv-scoring` (`.venv-scoring/Scripts/python.exe`).


In [1]:
from pathlib import Path
import sys
import os
import json
import pandas as pd
from IPython.display import display

ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents)
             if (path / 'configs' / 'config.yaml').is_file()
             and (path / 'outputs' / 'baseline.csv').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the csc173-midterm repository.')
os.chdir(ROOT)
if Path(sys.prefix).resolve() != (ROOT / '.venv-scoring').resolve():
    raise RuntimeError('Select the .venv-scoring notebook kernel, then restart.')
import hashlib

RUN = ROOT / 'outputs' / 'faithfulness' / 'run_01'
N = 618

print('Run folder:', RUN)
print('Kernel:', sys.executable)


Run folder: c:\Users\Yelfia\Documents\DeepLearning\MIDTERMPROJ\csc173-midterm\outputs\faithfulness\run_01
Kernel: c:\Users\Yelfia\Documents\DeepLearning\MIDTERMPROJ\csc173-midterm\.venv-scoring\Scripts\python.exe


## Function definitions


### Shared helper functions

Reading, validation, file saving, and resumption helpers.


In [2]:
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def read_csv(path):
    return pd.read_csv(path, dtype=str, keep_default_na=False, encoding='utf-8-sig')


def require(condition, message):
    if not condition:
        raise ValueError(message)


def nonblank(frame, columns):
    for column in columns:
        require(column in frame, f'Missing column: {column}')
        require(frame[column].map(lambda x: isinstance(x, str) and bool(x.strip())).all(),
                f'Blank or invalid text in {column}')


def reference_list(text):
    values = json.loads(text)
    require(isinstance(values, list) and len(values) > 0, 'Missing reference list')
    require(all(isinstance(x, str) and bool(x.strip()) for x in values), 'Invalid reference')
    return values


def atomic_csv(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.tmp')
    frame.to_csv(temporary, index=False, lineterminator='\n')
    os.replace(temporary, path)


def write_json(value, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2,
                                    allow_nan=False), encoding='utf-8')
    os.replace(temporary, path)


def manifest(path, value):
    # Reject changed inputs/configuration instead of reusing incompatible partial results.
    encoded = json.dumps(value, sort_keys=True, ensure_ascii=False, allow_nan=False)
    if path.exists():
        old = json.dumps(json.loads(path.read_text(encoding='utf-8')),
                         sort_keys=True, ensure_ascii=False, allow_nan=False)
        require(old == encoded, f'Protocol changed: {path}. Use a new run folder.')
    else:
        write_json(value, path)


def load_eval():
    frame = read_csv(RUN / 'evaluation_pairs.csv')
    nonblank(frame, ['sample_id', 'abstract', 'baseline_summary', 'human_tldrs'])
    require(len(frame) == N and frame.sample_id.is_unique, 'Expected 618 unique papers')
    frame.human_tldrs.map(reference_list)
    return frame


def validate_candidates(frame, evaluation):
    nonblank(frame, ['sample_id', 'candidate_id', 'abstract', 'candidate_summary'])
    require(len(frame) == N * 5, 'Expected 3090 candidate rows')
    require(set(frame.sample_id) == set(evaluation.sample_id), 'Candidate ID set differs')
    require(not frame.duplicated(['sample_id', 'candidate_id']).any(), 'Duplicate keys')
    for sample_id, group in frame.groupby('sample_id', sort=False):
        require(set(group.candidate_id.astype(str)) == set(map(str, range(5))),
                f'Candidate IDs must be 0..4: {sample_id}')
        normalized = group.candidate_summary.map(lambda text: ' '.join(text.split()))
        require(normalized.nunique() == 5, f'Non-distinct candidates: {sample_id}')
    joined = frame.merge(evaluation[['sample_id', 'abstract']], on='sample_id',
                         validate='many_to_one', suffixes=('', '_expected'))
    require((joined.abstract == joined.abstract_expected).all(), 'Wrong candidate source')


### `run_scoring()`

Score summaries using SummaCZS and AlignScore. 

### NOTE
This is a shared function with notebook 8. Apply any change to both notebooks for consistency.

#### Purpose and inputs

`run_scoring(kind)` compares each summary with its source abstract using two pretrained evaluators. Note that this function does not perform training nor does it rewrite the summaries, it performs inference.

The two output columns are `summac_score` and `alignscore_score`.

| Mode | Input file | Summary column | Unique row key | Output file |
|---|---|---|---|---|
| `'baseline'` | `evaluation_pairs.csv` | `baseline_summary` | `sample_id` | `baseline_scores.csv` |
| `'candidate'` | `generated_candidates.csv` | `candidate_summary` | `sample_id`, `candidate_id` | `candidate_scores.csv` |

A paper has one baseline but five retained candidates. Candidate IDs repeat across papers, so both identifiers are needed in candidate mode. The nested `score_table()` contains the actual procedure; the outer function calls it at the end.

#### Model settings and evidence

For the purposes of expediting the process, the function requires CUDA. To reproduce the entire pipeline using CPU, refer to SETUP_CPU.md.

1. SummaC 
- `model_name='vitc'`
- sentence granularity
- `op1='max'` and `op2='mean'`
- entailment-only settings (`use_ent=True`, `use_con=False`)
- Metric compares source and summary sentence content for support and aggregates those comparisons
- abstracts as `sources`

2. AlignScore 
- `roberta-base`
- local `AlignScore-base.ckpt`
- `evaluation_mode='nli_sp'`
- Neural calculations are handled by the installed packages
- abstracts as `contexts`

Summaries as `claims`. 

Both scores are expected within the range `[0, 1]`. Higher values indicates greater support according to the evaluator. The scales between the two evaluators are not interchangeable and thus, must be treated independently.

#### Batch processing and saving

The outer loop runs on a batch process of eight summary pairs at a time. 
1. `.iloc[start:start + 8]` selects positional slices
2. `.copy()` creates a separate table before scores are added

`torch.inference_mode()` avoids gradient bookkeeping during model inference
1. Checks sentence splitting
2. Requires one numeric output per input pair
3. Rejects NaN, infinity, or out-of-range scores
4. **metric failure stops the call instead of being replaced with a fabricated zero.**

The SummaC intermediate cache is cleared each batch
- local model references are deleted and garbage collection is requested at completion
- Downloaded model files remain on disk

#### Quality of Life

1. `transformers.AdamW = torch.optim.AdamW` allows PyTorch's AdamW class to become accessible. This is done by supplying a compatibility alias that matches older AlignScore code. This serves as a compatibility fix

2. The checkpoint existence check ensures AlignScore's trained file is available. 

3. `snapshot_download()` resolves needed metric model/tokenizer files and records cache snapshot names. Hugging Face offline flags are then set for subsequent cached loading; these remain active in that kernel.

4. The score manifest binds a score file to hashes of its input CSV and metric manifest. This helps avoid resuming on changed text or settings. 

5. If an output CSV already exists, the function verifies its keys, source/summary text, columns, and score values before trusting it. `completed = set(map(tuple, done[keys].to_numpy()))` converts saved identifiers into a lookup set. `pending` contains only rows whose keys are not in that set.

6. `atomic_csv()` saves the full accumulated table after each successful batch. Interruped batches will be discarded while previously saved batches will be reused.



In [3]:
def run_scoring(kind):
    """Score baseline or candidate summaries, saving each completed batch."""
    require(kind in {'baseline', 'candidate'}, 'Choose baseline or candidate scoring')
    import gc
    from pathlib import Path
    import numpy as np
    import pandas as pd
    import torch
    import transformers
    from huggingface_hub import snapshot_download

    transformers.AdamW = torch.optim.AdamW
    from alignscore import AlignScore
    from summac.model_summac import SummaCZS

    require(torch.cuda.is_available(),
            'CUDA is unavailable in this kernel. Install CUDA-enabled PyTorch in .venv-scoring and restart.')
    print('Scoring GPU:', torch.cuda.get_device_name(0), flush=True)
    PROTOCOL = {
        'device': 'cuda', 'outer_batch': 8, 'internal_batch': 8,
        'summac': {'variant': 'ZS', 'model_name': 'vitc', 'granularity': 'sentence',
                   'op1': 'max', 'op2': 'mean', 'use_ent': True, 'use_con': False,
                   'max_doc_sents': 100, 'max_input_length': 500},
        'alignscore': {'model': 'roberta-base', 'evaluation_mode': 'nli_sp'},
    }

    def score_table(kind):
        evaluation = load_eval()
        if kind == 'baseline':
            source = RUN / 'evaluation_pairs.csv'
            data = evaluation.copy()
            keys, text_column = ['sample_id'], 'baseline_summary'
        else:
            source = RUN / 'generated_candidates.csv'
            data = read_csv(source)
            validate_candidates(data, evaluation)
            keys, text_column = ['sample_id', 'candidate_id'], 'candidate_summary'
        nonblank(data, ['abstract', text_column])
        checkpoint = ROOT / 'checkpoints' / 'AlignScore-base.ckpt'
        require(checkpoint.is_file(), 'AlignScore base checkpoint is missing')

        # Resolve metric repositories once; record immutable cache snapshot IDs.
        # With HF_HUB_OFFLINE=1, these must already be cached.
        model_ids = ['roberta-base', 'tals/albert-xlarge-vitaminc-mnli']
        patterns = ['config.json', 'tokenizer*', 'vocab.*', 'merges.txt',
                    'special_tokens_map.json', 'added_tokens.json', 'spiece.model',
                    '*.safetensors', 'pytorch_model*.bin']
        snapshots = {name: Path(snapshot_download(name, allow_patterns=patterns)).name
                     for name in model_ids}
        # Load below from cache, avoiding a moving main branch during this process.
        import os
        os.environ['HF_HUB_OFFLINE'] = '1'
        import huggingface_hub.constants
        huggingface_hub.constants.HF_HUB_OFFLINE = True

        metric_record = {'protocol': PROTOCOL, 'snapshots': snapshots}
        manifest(RUN / 'metric_manifest.json', metric_record)
        output = RUN / f'{kind}_scores.csv'
        manifest(RUN / f'{kind}_score_manifest.json', {
            'input_sha256': sha256(source), 'metric_manifest_sha256': sha256(RUN / 'metric_manifest.json')})
        if output.exists():
            done = read_csv(output)
            require(set(done.columns) == set(data.columns) | {'summac_score', 'alignscore_score'},
                    'Unexpected partial score columns')
            require(not done.duplicated(keys).any(), 'Duplicate partial score keys')
            check = done.merge(data, on=keys, suffixes=('_saved', '_input'),
                               how='left', validate='one_to_one', indicator=True)
            require((check['_merge'] == 'both').all(), 'Unknown saved score key')
            for column in data.columns:
                if column not in keys:
                    require((check[column + '_saved'] == check[column + '_input']).all(),
                            f'Saved text/metadata changed: {column}')
            values = done[['summac_score', 'alignscore_score']].astype(float).to_numpy()
            require(np.isfinite(values).all() and ((values >= 0) & (values <= 1)).all(),
                    'Invalid saved scores')
        else:
            done = pd.DataFrame(columns=[*data.columns, 'summac_score', 'alignscore_score'])
        completed = set(map(tuple, done[keys].to_numpy()))
        pending = data[[tuple(row) not in completed for row in data[keys].to_numpy()]]
        if pending.empty:
            print(kind, 'already complete:', len(done))
            return

        summac = SummaCZS(model_name='vitc', granularity='sentence', op1='max', op2='mean',
            use_ent=True, use_con=False, device=PROTOCOL['device'],
            imager_load_cache=False, use_cache=False, max_doc_sents=100)
        align = AlignScore(model='roberta-base', batch_size=PROTOCOL['internal_batch'],
            device=PROTOCOL['device'], ckpt_path=str(checkpoint),
            evaluation_mode='nli_sp', verbose=False)
        for start in range(0, len(pending), PROTOCOL['outer_batch']):
            batch = pending.iloc[start:start + PROTOCOL['outer_batch']].copy()
            sources, summaries = batch.abstract.tolist(), batch[text_column].tolist()
            for source_text, summary in zip(sources, summaries):
                require(bool(summac.imager.split_sentences(source_text)), 'No source sentences')
                require(bool(summac.imager.split_sentences(summary)), 'No summary sentences')
            with torch.inference_mode():
                first = np.asarray(summac.score(sources, summaries,
                    batch_size=PROTOCOL['internal_batch'])['scores'], dtype=float)
                second = np.asarray(align.score(contexts=sources, claims=summaries), dtype=float)
            require(first.shape == second.shape == (len(batch),), 'Wrong score count')
            values = np.column_stack([first, second])
            require(np.isfinite(values).all() and ((values >= 0) & (values <= 1)).all(),
                    'Nonfinite or out-of-range scores; stop instead of inserting zeros')
            batch['summac_score'], batch['alignscore_score'] = first, second
            done = pd.concat([done, batch], ignore_index=True)
            atomic_csv(done, output)
            print(kind, len(done), '/', len(data), flush=True)
            # The package's batch path retains image caches even with use_cache=False.
            summac.imager.cache.clear()
        require(len(done) == len(data), 'Incomplete scores')
        del align, summac
        gc.collect()
    score_table(kind)


## Execution and results


In [4]:
evaluation = load_eval()
run_scoring('baseline')
baseline_path = RUN / 'baseline_scores.csv'
if baseline_path.exists():
    baseline = read_csv(baseline_path)
    print(f'Saved baseline rows: {len(baseline)}/618')
    display(baseline[['sample_id', 'summac_score', 'alignscore_score']].head())


c:\Users\Yelfia\Documents\DeepLearning\MIDTERMPROJ\csc173-midterm\.venv-scoring\Lib\site-packages\lightning_fabric\__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)


Scoring GPU: NVIDIA GeForce RTX 5050 Laptop GPU


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Lightning automatically upgraded your loaded checkpoint from v1.7.7 to v1.9.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint --file c:\Users\Yelfia\Documents\DeepLearning\MIDTERMPROJ\csc173-midterm\checkpoints\AlignScore-base.ckpt`
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
c:\Users\Yelfia\Documents\DeepLearning\MIDTERMPROJ\csc173-midterm\.venv-scoring\Lib\site-packages\pytorch_lightning\core\saving.py:255: UserWarning: Found

baseline 8 / 618
baseline 16 / 618
baseline 24 / 618
baseline 32 / 618
baseline 40 / 618
baseline 48 / 618
baseline 56 / 618
baseline 64 / 618
baseline 72 / 618
baseline 80 / 618
baseline 88 / 618
baseline 96 / 618
baseline 104 / 618
baseline 112 / 618
baseline 120 / 618
baseline 128 / 618
baseline 136 / 618
baseline 144 / 618
baseline 152 / 618
baseline 160 / 618
baseline 168 / 618
baseline 176 / 618
baseline 184 / 618
baseline 192 / 618
baseline 200 / 618
baseline 208 / 618
baseline 216 / 618
baseline 224 / 618
baseline 232 / 618
baseline 240 / 618
baseline 248 / 618
baseline 256 / 618
baseline 264 / 618
baseline 272 / 618
baseline 280 / 618
baseline 288 / 618
baseline 296 / 618
baseline 304 / 618
baseline 312 / 618
baseline 320 / 618
baseline 328 / 618
baseline 336 / 618
baseline 344 / 618
baseline 352 / 618
baseline 360 / 618
baseline 368 / 618
baseline 376 / 618
baseline 384 / 618
baseline 392 / 618
baseline 400 / 618
baseline 408 / 618
baseline 416 / 618
baseline 424 / 618
baseli

,sample_id,summac_score,alignscore_score
0,SJ1Xmf-Rb,0.732421875,0.9945850968360901
1,S1xzyhR9Y7,0.9912109375,0.9935088157653809
2,HJDUjKeA-,0.99365234375,0.9911770820617676
3,BJgLg3R9KQ,0.109130859375,0.9904725551605225
4,BklpOo09tQ,0.9951171875,0.9837444424629211
